In [ ]:
!pip install transformers torch accelerate bitsandbytes einops gradio typing langchain-community pypdf faiss-cpu sentence-transformers huggingface_hub langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 9.9 MB/s eta 0:00:00
   ━

In [ ]:
from transformers import AutoTokenizer, TextIteratorStreamer, AutoModelForCausalLM
import transformers
import torch
from torch import bfloat16
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from threading import Thread
import asyncio
from huggingface_hub import notebook_login
import gradio as gr
from typing import Iterator
import os

In [ ]:
notebook_login()

In [ ]:
model_id = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.chat_template = """{% for message in messages %}
# {{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}
# {% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"""

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [ ]:
MAX_MAX_NEW_TOKENS = 2048
DEFAULT_MAX_NEW_TOKENS = 1024
MAX_INPUT_TOKEN_LENGTH = int(os.getenv("MAX_INPUT_TOKEN_LENGTH", "4096"))

def process_pdfs_sync(files):
    """Process PDF files and create FAISS index"""
    if not files:
        return None

    documents = []
    for file_path in files:
        loader = PyPDFLoader(file_path.name)
        documents.extend(loader.load())

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=128,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": False}
    )

    return FAISS.from_documents(chunks, embeddings)


In [ ]:
css = """
.spinner {
    display: inline-block;
    width: 50px;
    height: 50px;
    border: 5px solid #f3f3f3;
    border-radius: 50%;
    border-top: 5px solid #3498db;
    animation: spin 1s linear infinite;
}
@keyframes spin {
    0% { transform: rotate(0deg); }
    100% { transform: rotate(360deg); }
}
"""

In [ ]:
async def process_pdfs(files, status):
    """Process PDFs and reset chat history"""
    # Initial yield with 4 outputs
    yield (
        "<div style='padding: 20px; text-align: center;'><div class='spinner'></div><br>Processing PDFs...</div>",
        gr.update(interactive=False),  # question_input
        gr.update(interactive=False),  # submit_btn
        None  # vector_store (placeholder)
    )

    loop = asyncio.get_event_loop()
    vector_store = await loop.run_in_executor(None, process_pdfs_sync, files)

    # Final yield with 4 outputs
    yield (
        "<div style='color: green; padding: 20px;'>✅ Documents ready</div>",
        gr.update(interactive=True),  # question_input
        gr.update(interactive=True),  # submit_btn
        vector_store  # actual vector_store
    )

In [ ]:
def format_prompt(question: str, context: str, history: list):
    """Create conversation prompt with history using proper chat template"""
    conversation = []

    # System message with context
    if context:
        conversation.append({
            "role": "system",
            "content": (
                "You are Medical AI assistant. Be cheerful and hopeful always while responding to the user. "
                "Don't use hate speech. Keep responses short, crisp and concise. "
                "Only respond to medical-related queries. For non-medical questions, respond: "
                "'Sorry, that question is not related to the medical domain. Please ask a health-related question.'\n"
                f"Context: {context}"
            )
        })
    else:  # Fallback system message
        conversation.append({
            "role": "system",
            "content": (
                "You are a helpful Medical AI assistant. Provide cheerful, concise health advice. "
                "Only answer medical questions. Redirect non-medical queries appropriately."
            )
        })

    # Add conversation history (last 5 exchanges)
    for user_msg, bot_resp in history[-5:]:
        conversation.extend([
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": bot_resp}
        ])

    # Add current question
    conversation.append({"role": "user", "content": question})

    # Apply chat template with generation prompt
    return tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True  # Crucial for triggering assistant response
    )


In [ ]:
def generate(question: str, vector_store: FAISS, chat_history: list):
    """Generate response with RAG and history"""
    context = ""
    if vector_store:
        retriever = vector_store.as_retriever(search_kwargs={"k": 3})
        relevant_docs = retriever.invoke(question)
        context = "\n".join([doc.page_content for doc in relevant_docs])

    # Create temporary history copy
    temp_history = chat_history.copy()
    # Add new empty response
    temp_history.append((question, ""))

    # Create prompt with history
    prompt = format_prompt(question, context, chat_history)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    if input_ids.shape[1] > MAX_INPUT_TOKEN_LENGTH:
        input_ids = input_ids[:, -MAX_INPUT_TOKEN_LENGTH:]
        gr.Warning(f"Trimmed input to {MAX_INPUT_TOKEN_LENGTH} tokens")

    streamer = TextIteratorStreamer(tokenizer, timeout=10.0, skip_prompt=True, skip_special_tokens=True)
    generate_kwargs = {
        "input_ids": input_ids,
        "streamer": streamer,
        "max_new_tokens": 1024,
        "do_sample": True,
        "top_p": 0.9,
        "top_k": 50,
        "temperature": 0.6,
        "repetition_penalty": 1.2,
    }

    t = Thread(target=model.generate, kwargs=generate_kwargs)
    t.start()

    output = []
    for token in streamer:
        output.append(token)
        # Update the last response in temporary history
        temp_history[-1] = (question, "".join(output))
        # Yield both the chat interface and history state
        yield temp_history, temp_history

    # Final update to keep only last 5 exchanges
    final_history = temp_history[-5:]
    yield final_history, final_history

In [ ]:
# tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [ ]:
with gr.Blocks(css=css) as demo:
    vector_store = gr.State()
    chat_history = gr.State([])

    with gr.Row():
        status = gr.HTML("ℹ️ Upload PDFs to begin")

    with gr.Row():
        pdf_upload = gr.UploadButton(
            "📁 Upload PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )

    with gr.Row():
        question_input = gr.Textbox(
            label="Your Question",
            placeholder="Type your question...",
            interactive=False
        )
        submit_btn = gr.Button("Ask", interactive=False)

    chatbot = gr.Chatbot(height=500)
    clear_btn = gr.Button("Clear History")

    # PDF processing
    pdf_upload.upload(
    process_pdfs,
    inputs=[pdf_upload, status],
    outputs=[status, question_input, submit_btn, vector_store]  # 4 outputs
)

    # Chat handling
    submit_btn.click(
    generate,
    [question_input, vector_store, chat_history],
    [chatbot, chat_history]  # Now both outputs get the history list
)

    question_input.submit(
        lambda: gr.update(value=""),  # Clear input
        outputs=[question_input]
    )

    clear_btn.click(
        lambda: [],
        outputs=[chat_history]
    )

demo.queue().launch(debug = True)

<ipython-input-11-07d04104750d>:23: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500)


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a0124fb2fc6a027969.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


<ipython-input-5-489b964da4eb>:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a0124fb2fc6a027969.gradio.live
